In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("EXCHANGERATE_API_KEY")

In [2]:
import time
import requests
import pandas as pd


API_KEY     = os.getenv("EXCHANGERATE_API_KEY")       # free key from exchangerate.host
if not API_KEY:
    raise ValueError("EXCHANGERATE_API_KEY not set")

OUTPUT_FILE = "usd_lkr_2013_2019.csv"


def fetch_year(year: int) -> list:
    url = "https://api.exchangerate.host/timeframe"
    params = {
        "access_key":  API_KEY,
        "start_date":  f"{year}-01-01",
        "end_date":    f"{year}-12-31",
        "base":        "USD",
        "symbols":     "LKR",
    }
    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()
    data = response.json()

    rows = []
    if data.get("success") and data.get("quotes"):
        for date_str, pairs in data["quotes"].items():
            rate = pairs.get("USDLKR")
            if rate:
                rows.append({"date": date_str, "usd_lkr": rate})
    return rows


all_rows = []

for year in range(2013, 2020):
    print(f"Fetching {year} ...", end=" ")
    try:
        rows = fetch_year(year)
        all_rows.extend(rows)
        print(f"{len(rows)} days fetched ✓")
    except Exception as e:
        print(f"ERROR: {e}")
    time.sleep(0.5)



df = pd.DataFrame(all_rows)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("\n── Summary ──────────────────────────────")
print(f"  Total days   : {len(df):,}")
print(f"  Date range   : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Rate range   : {df['usd_lkr'].min():.2f} → {df['usd_lkr'].max():.2f} LKR")
print(f"  Mean rate    : {df['usd_lkr'].mean():.2f} LKR")
print()
print(df.head(10).to_string(index=False))

df.to_csv(OUTPUT_FILE, index=False)

Fetching 2013 ... 365 days fetched ✓
Fetching 2014 ... 365 days fetched ✓
Fetching 2015 ... 365 days fetched ✓
Fetching 2016 ... 366 days fetched ✓
Fetching 2017 ... 365 days fetched ✓
Fetching 2018 ... 365 days fetched ✓
Fetching 2019 ... 365 days fetched ✓

── Summary ──────────────────────────────
  Total days   : 2,556
  Date range   : 2013-01-01 → 2019-12-31
  Rate range   : 125.39 → 183.00 LKR
  Mean rate    : 147.91 LKR

      date    usd_lkr
2013-01-01 127.640174
2013-01-02 127.431517
2013-01-03 127.457218
2013-01-04 127.408818
2013-01-05 127.401118
2013-01-06 127.399318
2013-01-07 127.401218
2013-01-08 127.224609
2013-01-09 126.490904
2013-01-10 126.178581


In [3]:
import pandas as pd

df = pd.read_csv("usd_lkr_2013_2019.csv")
df.head()

,date,usd_lkr
0,2013-01-01,127.640174
1,2013-01-02,127.431517
2,2013-01-03,127.457218
3,2013-01-04,127.408818
4,2013-01-05,127.401118


In [4]:
data = pd.read_csv(r"Combined_data - Main - Combined_data.csv")
data.head()

,Year_Week,year,week,location,code,vegetable_type,price,no_of_holidays,zone,seasonality,lanka_auto_diesel_price,mean_apparent_temperature,rain_sum,usd_exchange_rate
0,2013-w1,2013,w1,Thambuththegama,323,PUMPKIN,37.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-w1,2013,w1,Kandy,323,PUMPKIN,42.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-w1,2013,w1,Dambulla,323,PUMPKIN,48.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-w1,2013,w1,Puttalam,323,PUMPKIN,52.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-w1,2013,w1,Nuwaraeliya,323,PUMPKIN,54.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df['Date'] = pd.to_datetime(df['date'])
df['week_start'] = df['Date'] - pd.to_timedelta((df['Date'].dt.weekday - 4) % 7, unit='D')

weekly_avg = (
    df.groupby('week_start')['usd_lkr']
    .mean()
    .reset_index()
)

weekly_avg['year'] = weekly_avg['week_start'].dt.isocalendar().year
weekly_avg['week_num'] = weekly_avg['week_start'].dt.isocalendar().week

rate_lookup = weekly_avg.set_index(['year', 'week_num'])['usd_lkr'].to_dict()
data['week_num'] = data['week'].str.extract(r'(\d+)').astype(int)

data['usd_lkr'] = data.apply(
    lambda row: rate_lookup.get((row['year'], row['week_num']), float('nan')),
    axis=1
)

data.drop(columns=['week_num'], inplace=True)
data.head()

,Year_Week,year,week,location,code,vegetable_type,price,no_of_holidays,zone,seasonality,lanka_auto_diesel_price,mean_apparent_temperature,rain_sum,usd_exchange_rate,usd_lkr
0,2013-w1,2013,w1,Thambuththegama,323,PUMPKIN,37.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,127.072081
1,2013-w1,2013,w1,Kandy,323,PUMPKIN,42.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,127.072081
2,2013-w1,2013,w1,Dambulla,323,PUMPKIN,48.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,127.072081
3,2013-w1,2013,w1,Puttalam,323,PUMPKIN,52.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,127.072081
4,2013-w1,2013,w1,Nuwaraeliya,323,PUMPKIN,54.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,127.072081


In [6]:
data.to_csv("exchange_rates_added.csv")